# Lab 1 at home · Two tables of your own in DuckDB

**Due:** Sunday, October 25 at 11:59 PM · **Submit:** your repo URL on CourseWorks

In class you loaded one table (penguins), found a problem and fixed it. Now do the same with **two related tables you choose**. In Session 2 we'll join them, so pick two that can be connected.

You will:
1. Pick two related public datasets and upload them
2. Load both into DuckDB and check what DuckDB decided about each column
3. Find the column the two tables share (the **key**) and check how well they match
4. Write five SQL queries that help explain your data
5. Write observations and a dataset card for each table

## 1. Pick two related tables

**Related** means they share a column you could match rows on: an ID, a name, a ZIP code, a country code, a date.

**Example pairs**
- **World Bank:** any indicator download (e.g. GDP per country per year) comes with a second file of country details (region, income group). The shared column is the country code.
- **Our World in Data:** two charts for the same countries, e.g. life expectancy and health spending. Shared columns: country and year.
- **NYC Open Data:** 311 complaints and population by ZIP code, or the street tree census and neighborhood boundaries. Shared column: ZIP code or neighborhood.
- **Your own idea:** any two public tables that share a column.

**Rules**
- **Public, open data only.** Your repo is public, and later this data goes to a free AI tier. Nothing from work, nothing personal, nothing behind a login.
- **CSV format.** If a file is Excel, open it and save it as CSV first.
- **Each file under 100 MB.** Between 1,000 and 1,000,000 rows is a good size for the main table; the second can be small.
- **Reuse allowed.** Check each license or terms page, and note it in your dataset card.

**Places to look:** [NYC Open Data](https://opendata.cityofnewyork.us) · [data.gov](https://data.gov) · [World Bank Open Data](https://data.worldbank.org) · [Our World in Data](https://ourworldindata.org) · [Kaggle Datasets](https://www.kaggle.com/datasets) (check each license)

## 2. Upload them

1. Download both CSVs to your computer.
2. In this Codespace, drag both files into the **`data`** folder in the file list on the left.
3. Rename each to something short with no spaces, e.g. `gdp.csv` and `countries.csv` (right-click → **Rename**).
4. Fill in the cell below and run it. Table names should be short, lowercase, no spaces: you'll type them in your SQL.

In [13]:
FILE_A  = "Queens_HPD_Complaints_Problems_2025.csv"     # <- your main table's file name
TABLE_A = "Violations"                   # <- a short name for it, e.g. "gdp"

FILE_B  = "Queens_HPD_Violations_2025.csv"   # <- the related table's file name
TABLE_B = "Complaints"                 # <- a short name for it, e.g. "countries"

In [14]:
from pathlib import Path
import duckdb

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
DATA_A = ROOT / "data" / FILE_A
DATA_B = ROOT / "data" / FILE_B

missing = [p.name for p in (DATA_A, DATA_B) if not p.exists()]
if missing:
    print("Can't find:", ", ".join(missing))
    print("Files in data/ right now:")
    for f in sorted((ROOT / "data").iterdir()):
        print("  ", f.name)
else:
    con = duckdb.connect(str(ROOT / "lab1_home.duckdb"))
    for p in (DATA_A, DATA_B):
        print(f"Found data/{p.name} ({p.stat().st_size / 1_000_000:.1f} MB)")
    print("Connected.")

Found data/Queens_HPD_Complaints_Problems_2025.csv (71.6 MB)
Found data/Queens_HPD_Violations_2025.csv (59.9 MB)
Connected.


## 3. First look

Preview the first 10 rows of each raw file before loading.

In [25]:
con.sql(f"SELECT * FROM read_csv('{DATA_A}') LIMIT 10")

┌─────────────────────┬────────────┬──────────────┬─────────────┬─────────┬──────────────┬──────────────────┬───────────┬───────┬───────┬───────────┬─────────────────┬───────────────┬──────────────────┬─────────────────────┬──────────────────────┬─────────────────┬──────────────────────────┬──────────────────┬───────────────────────┬────────────────┬─────────────────────┬─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬────────────────────────

In [26]:
con.sql(f"SELECT * FROM read_csv('{DATA_B}') LIMIT 10")

┌─────────────┬────────────┬────────────────┬────────┬─────────┬─────────────┬────────────────┬─────────────────┬────────────┬────────────┬───────┬───────────┬───────┬───────┬───────┬─────────┬─────────────────────┬─────────────────────┬───────────────────────┬───────────────────────┬──────────────────┬──────────────────┬─────────────────────┬─────────────┬──────────┬────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬─────────────────────┬─────────────────┬─────────────────────┬─────────────────────┬──────────┬─────────────────┬───────────────┬───────────┬────────────┬────────────────┬─────────────────┬─────────────┬─────────┬────────────┬───────────────────┐
│ violationid │ buildingid │ registrationid │ boroid │  boro   │ housenumber │ lowhousenumber │ highhousenumber │ streetname │ streetcode │  zip  │ apartment 

## 4. Load both and check the types

Load each file into a table, then `DESCRIBE` it. Compare each column's type with what it *should* be. A number or date column stored as `VARCHAR` (text) is the same kind of problem you fixed in class.

In [27]:
for table, path in ((TABLE_A, DATA_A), (TABLE_B, DATA_B)):
    con.sql(f"CREATE OR REPLACE TABLE {table} AS SELECT * FROM read_csv('{path}')")
    rows = con.sql(f"SELECT count(*) FROM {table}").fetchone()[0]
    print(f"{table}: {rows:,} rows loaded")

Violations: 107,497 rows loaded
Complaints: 106,400 rows loaded


In [28]:
con.sql(f"DESCRIBE {TABLE_A}")

┌──────────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│       column_name        │ column_type │  null   │   key   │ default │  extra  │
│         varchar          │   varchar   │ varchar │ varchar │ varchar │ varchar │
├──────────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ received_date            │ TIMESTAMP   │ YES     │ NULL    │ NULL    │ NULL    │
│ problem_id               │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ complaint_id             │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ building_id              │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ borough                  │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ house_number             │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ street_name              │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ post_code                │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ bl

In [29]:
con.sql(f"DESCRIBE {TABLE_B}")

┌─────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│   column_name   │ column_type │  null   │   key   │ default │  extra  │
│     varchar     │   varchar   │ varchar │ varchar │ varchar │ varchar │
├─────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ violationid     │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ buildingid      │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ registrationid  │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ boroid          │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ boro            │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ housenumber     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ lowhousenumber  │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ highhousenumber │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ streetname      │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ streetcode      │ BIGINT      │ YES 

`SUMMARIZE` profiles every column at once: min, max, how many distinct values, and `null_percentage` (how much is missing).

In [30]:
con.sql(f"SUMMARIZE {TABLE_A}")

┌──────────────────────────┬─────────────┬─────────────────────┬─────────────────────┬───────────────┬────────────────────────────┬──────────────────────┬────────────────────────────┬────────────────────────────┬────────────────────────────┬────────┬─────────────────┐
│       column_name        │ column_type │         min         │         max         │ approx_unique │            avg             │         std          │            q25             │            q50             │            q75             │ count  │ null_percentage │
│         varchar          │   varchar   │       varchar       │       varchar       │     int64     │          varchar           │       varchar        │          varchar           │          varchar           │          varchar           │ int64  │  decimal(9,2)   │
├──────────────────────────┼─────────────┼─────────────────────┼─────────────────────┼───────────────┼────────────────────────────┼──────────────────────┼────────────────────────────┼──────────

In [31]:
con.sql(f"SUMMARIZE {TABLE_B}")

┌─────────────────┬─────────────┬───────────────────┬───────────────────────┬───────────────┬────────────────────┬──────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────┬─────────────────┐
│   column_name   │ column_type │        min        │          max          │ approx_unique │        avg         │         std          │        q25         │        q50         │        q75         │ count  │ null_percentage │
│     varchar     │   varchar   │      varchar      │        varchar        │     int64     │      varchar       │       varchar        │      varchar       │      varchar       │      varchar       │ int64  │  decimal(9,2)   │
├─────────────────┼─────────────┼───────────────────┼───────────────────────┼───────────────┼────────────────────┼──────────────────────┼────────────────────┼────────────────────┼────────────────────┼────────┼─────────────────┤
│ violationid     │ BIGINT      │ 135101            │ 18549146              │         96

## 5. Fix what loaded wrong (if anything)

If every type looks right, write "No fixes needed" in the text cell after the next one and skip ahead.

Otherwise, open the CSV (double-click it in the `data` folder) and find **how missing values are written** in the columns that came in as text. Edit the `nullstr` list below to match, and run it for the table that needs fixing.

In [32]:
# Change TABLE_A / DATA_A to TABLE_B / DATA_B if it's the second table that needs fixing
con.sql(f"""
    CREATE OR REPLACE TABLE {TABLE_A} AS
    SELECT * FROM read_csv('{DATA_A}', nullstr = ['NA', 'N/A', ''])
""")
con.sql(f"DESCRIBE {TABLE_A}")

┌──────────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│       column_name        │ column_type │  null   │   key   │ default │  extra  │
│         varchar          │   varchar   │ varchar │ varchar │ varchar │ varchar │
├──────────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ received_date            │ TIMESTAMP   │ YES     │ NULL    │ NULL    │ NULL    │
│ problem_id               │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ complaint_id             │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ building_id              │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ borough                  │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ house_number             │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ street_name              │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ post_code                │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ bl

In [33]:
# Change TABLE_A / DATA_A to TABLE_B / DATA_B if it's the second table that needs fixing
con.sql(f"""
    CREATE OR REPLACE TABLE {TABLE_B} AS
    SELECT * FROM read_csv('{DATA_B}', nullstr = ['NA', 'N/A', ''])
""")
con.sql(f"DESCRIBE {TABLE_B}")

┌─────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│   column_name   │ column_type │  null   │   key   │ default │  extra  │
│     varchar     │   varchar   │ varchar │ varchar │ varchar │ varchar │
├─────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ violationid     │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ buildingid      │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ registrationid  │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ boroid          │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ boro            │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ housenumber     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ lowhousenumber  │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ highhousenumber │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ streetname      │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ streetcode      │ BIGINT      │ YES 

**What did you fix, and how did you know?** (Double-click to edit.)

No additional fixes needed. DuckDB correctly identified the main numeric, categorical, and geographic coordinate fields.

## 6. Find the key: how do the two tables connect?

The **key** is the column both tables share. Type its name in each table below. It may be spelled differently in each, e.g. `Country Code` in one and `iso3` in the other.

The next cell counts the distinct key values in each table and how many of them appear in both. (It treats both as text, so a ZIP code stored as a number still matches one stored as text.)

In [38]:
KEY_A = "building_id"   # <- the shared column's name in your main table
KEY_B = "buildingid"   # <- its name in the second table

In [39]:
con.sql(f"""
    WITH a AS (SELECT DISTINCT CAST("{KEY_A}" AS VARCHAR) AS k FROM {TABLE_A}),
         b AS (SELECT DISTINCT CAST("{KEY_B}" AS VARCHAR) AS k FROM {TABLE_B})
    SELECT
        (SELECT count(*) FROM a)                           AS keys_in_a,
        (SELECT count(*) FROM b)                           AS keys_in_b,
        (SELECT count(*) FROM a WHERE k IN (SELECT k FROM b)) AS keys_in_both
""")

┌───────────┬───────────┬──────────────┐
│ keys_in_a │ keys_in_b │ keys_in_both │
│   int64   │   int64   │    int64     │
├───────────┼───────────┼──────────────┤
│     11962 │     20190 │         6642 │
└───────────┴───────────┴──────────────┘

**Describe the relationship** (double-click to edit):

- **Key column:** in the main table it's called building_id, in the second table uses buildingid.
- **How they relate:** does one row in the second table match *one* row in the main table, or *many*? (e.g. one country has many GDP rows, one per year)
   One building can have multiple complaints and multiple violations, so the same building ID can appear many times in both tables.
- **How well they match:** how many keys appear in only one table? Give one example and a guess at why (spelling, codes vs. names, regions mixed in with countries...)
   There are 6,642 building IDs in both tables. Another 5,320 appear only in the complaints table, and 13,548 appear only in the violations table. This might be because some buildings have complaints but no recorded violations, while others have violations without complaints in our dataset.


## 7. Five queries that explain your data

Write one query of each kind. Under each, double-click the text cell and write one line: **what does this result tell us?** Use either table.

Tip: if a column name has spaces or capitals, wrap it in double quotes: `"Country Name"`. Start from a query in the in-class notebook and change one thing.

### Query 1 · Size and shape
*How many rows, and over what time period or area?* Useful SQL: `count(*), min(), max()`

In [42]:
con.sql(f"""
    --How many records are in the NYC Housing Maintenance Code Complaints and Problems dataset, and what time period and geographic areas does it cover?
    SELECT
        COUNT(*) AS total_records,
        MIN(received_date) AS earliest_date,
        MAX(received_date) AS latest_date,
        COUNT(DISTINCT borough) AS borough_count
    FROM {TABLE_A}
""")

┌───────────────┬─────────────────────┬─────────────────────┬───────────────┐
│ total_records │    earliest_date    │     latest_date     │ borough_count │
│     int64     │      timestamp      │      timestamp      │     int64     │
├───────────────┼─────────────────────┼─────────────────────┼───────────────┤
│        107497 │ 2025-01-01 00:34:41 │ 2025-12-31 23:37:45 │             1 │
└───────────────┴─────────────────────┴─────────────────────┴───────────────┘

**What this tells us:** 
The dataset contains 107,497 housing problem records from Queens in 2025, covering the full year from January 1 to December 31.

### Query 2 · Filter
*Which rows meet a condition you care about?* Useful SQL: `WHERE`

In [43]:
con.sql(f"""
    -- How many housing problems related to heat or hot water were reported in Queens in 2025?
    SELECT
        problem_code,
        COUNT(*) AS problem_count
    FROM {TABLE_A}
    WHERE problem_code IS NOT NULL
    GROUP BY problem_code
    ORDER BY problem_count DESC
    LIMIT 10
""")

┌──────────────────────────┬───────────────┐
│       problem_code       │ problem_count │
│         varchar          │     int64     │
├──────────────────────────┼───────────────┤
│ NO HEAT                  │         27611 │
│ NO HEAT AND NO HOT WATER │          9361 │
│ BROKEN OR MISSING        │          5693 │
│ NO HOT WATER             │          5134 │
│ ROACHES                  │          4580 │
│ MICE                     │          3748 │
│ AT WALL OR CEILING       │          3556 │
│ CHIPPED/PEELING/FLAKING  │          2675 │
│ NO WATER                 │          1821 │
│ ACCUMULATION             │          1596 │
└──────────────────────────┴───────────────┘
  10 rows                        2 columns

**What this tells us:** 
Heating problems are the most common issues in the dataset. "No Heat" has the highest number of records (27,611), followed by "No Heat and No Hot Water" (9,361). Other common problems include broken facilities, pests, and water issues.

### Query 3 · Group
*How does a count or average differ across categories?* Useful SQL: `GROUP BY`

In [45]:
con.sql(f"""
    -- How many housing violations are recorded in each violation class in the NYC Housing Maintenance Code Violations dataset?
    SELECT
        "class" AS violation_class,
        COUNT(*) AS violation_count
    FROM {TABLE_B}
    GROUP BY "class"
    ORDER BY violation_count DESC
    LIMIT 10
""")

┌─────────────────┬─────────────────┐
│ violation_class │ violation_count │
│     varchar     │      int64      │
├─────────────────┼─────────────────┤
│ B               │           40032 │
│ A               │           34175 │
│ C               │           32193 │
└─────────────────┴─────────────────┘

**What this tells us:** 
Class B violations are the most common, with 40,032 records, followed by Class A (34,175) and Class C (32,193). Although Class C represents the most serious violations, it still accounts for a substantial share of the records.

### Query 4 · Rank
*What are the top or bottom 10 of something?* Useful SQL: `ORDER BY ... LIMIT 10`

In [46]:
con.sql(f"""
    -- Which 10 buildings have the highest number of reported housing problems in the NYC Housing Maintenance Code Complaints and Problems dataset?
    SELECT
        building_id,
        COUNT(*) AS problem_count
    FROM {TABLE_A}
    WHERE building_id IS NOT NULL
    GROUP BY building_id
    ORDER BY problem_count DESC
    LIMIT 10
""")

┌─────────────┬───────────────┐
│ building_id │ problem_count │
│    int64    │     int64     │
├─────────────┼───────────────┤
│      694083 │           976 │
│      411605 │           926 │
│      569090 │           595 │
│      665250 │           589 │
│      428710 │           578 │
│      424613 │           507 │
│      421465 │           445 │
│      520396 │           367 │
│      642725 │           360 │
│      629716 │           340 │
└─────────────┴───────────────┘
  10 rows           2 columns

**What this tells us:** 
Housing problems are concentrated in a small number of buildings. Building 694083 has the most reported problems (976), followed by Building 411605 (926). This suggests that some buildings experience much more frequent maintenance issues than others.

### Query 5 · Quality check
*What's missing, duplicated or impossible?* Useful SQL: `count(*) - count(column), count(DISTINCT ...)`

In [47]:
con.sql(f"""
    -- How many records have missing building IDs or duplicate problem IDs?
    SELECT
        COUNT(*) AS total_records,
        COUNT(*) - COUNT(building_id) AS missing_building_ids,
        COUNT(*) - COUNT(problem_id) AS missing_problem_ids,
        COUNT(problem_id) - COUNT(DISTINCT problem_id) AS duplicate_problem_ids
    FROM {TABLE_A}
""")

┌───────────────┬──────────────────────┬─────────────────────┬───────────────────────┐
│ total_records │ missing_building_ids │ missing_problem_ids │ duplicate_problem_ids │
│     int64     │        int64         │        int64        │         int64         │
├───────────────┼──────────────────────┼─────────────────────┼───────────────────────┤
│        107497 │                    0 │                   0 │                     0 │
└───────────────┴──────────────────────┴─────────────────────┴───────────────────────┘

**What this tells us:** 
All 107,497 records have valid entries for building ID and problem ID, with no missing values or duplicate problem IDs. This suggests that the dataset is well structured for identifying individual problems and connecting records across tables.

## 8. Three observations

Specific things you noticed that you'd want to check before trusting this data. Point to numbers.

1. Heating is the most common housing problem in the complaints dataset, with 27,611 "No Heat" records. I would want to check whether these problems are concentrated in certain months or buildings.
2. Some buildings have unusually high numbers of reported problems. For example, Building 694083 has 976 records, while Building 411605 has 926. It would be useful to check whether these are large buildings or have repeated maintenance issues.
3. The two datasets share 6,642 building IDs, but many buildings appear in only one table. I would want to check whether this is related to differences in reporting, inspections, or the time periods covered by each dataset.

**One question you'd want a non-technical person to be able to ask of this data:**
Which buildings in my neighborhood have the most housing maintenance problems, and what kinds of issues are reported most often?



## 9. Dataset cards

| | Main table | Second table |
|---|---|---|
| **Dataset name** |Queens_HPD_Complaints_Problems_2025|Queens_HPD_Violations_2025|
| **Source (link)** |https://data.cityofnewyork.us/Housing-Development/Housing-Maintenance-Code-Complaints-and-Problems/ygpa-z7cr/about_data|https://data.cityofnewyork.us/Housing-Development/Housing-Maintenance-Code-Violations/wvxf-dwi5/about_data|
| **Who collected it, and why** |The NYC Department of Housing Preservation and Development (HPD) records public complaints submitted through 311, Code Enforcement Borough Offices, or the internet. The data documents reported housing-maintenance problems and supports complaint handling and enforcement of housing standards.|HPD records violations of the NYC Housing Maintenance Code or NYS Multiple Dwelling Law, generally verified by housing inspectors. The data supports enforcement, required corrections, and monitoring of violation status. Class I violations are generally issued administratively.|
| **License or terms of use** |NYC Open Data Terms of Use; NYC.gov Terms of Use and Privacy Policy also apply. No named licence is specified in the dataset metadata.|NYC Open Data Terms of Use; NYC.gov Terms of Use and Privacy Policy also apply. No named licence is specified in the dataset metadata.|
| **Rows × columns** |107,497 × 33|106,400 × 41|
| **Time period covered** |2025-01-01 to 2025-12-31|2025-01-01 to 2025-12-31|
| **How often it's updated** |Daily for the official source dataset|Daily for the official source dataset|

## 10. Submit

1. Run the cell below to close the connection.
2. Fill in the lifecycle map of your data in `labs/lab1.md`.
3. **Save this notebook (Cmd+S)**, then commit and push: **Source Control** → message `Lab 1` → **Commit** → **Sync Changes**.
4. Check on GitHub that both CSVs, both notebooks and `labs/lab1.md` are there, with results visible.
5. Paste your repo URL into the Lab 1 assignment on CourseWorks.

In [48]:
con.close()
print("Done. Save, commit, push, and submit your repo URL.")

Done. Save, commit, push, and submit your repo URL.
